<a href="https://colab.research.google.com/github/aman-verma02/Network-Security-System/blob/main/Colab-Google/Autoencoder_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/aman-verma02/Network-Security-System.git
%cd Network-Security-System
!pip install -r requirements.txt

Mounted at /content/drive
Cloning into 'Network-Security-System'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (273/273), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 273 (delta 106), reused 208 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (273/273), 5.65 MiB | 16.21 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/Network-Security-System
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.6/818.6 kB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.8

In [ ]:
import shutil

# Copy your 3 existing models into the cloned repo
shutil.copy('/content/drive/MyDrive/CICIDS_Dataset/model/model.pkl', 'final_model/model.pkl')
shutil.copy('/content/drive/MyDrive/CICIDS_Dataset/model/preprocessor.pkl', 'final_model/preprocessor.pkl')
shutil.copy('/content/drive/MyDrive/CICIDS_Dataset/model/feature_names.pkl', 'final_model/feature_names.pkl')

'final_model/feature_names.pkl'

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
import pickle

# ── Step 1: Load cleaned CSV ──
df = pd.read_csv('/content/drive/MyDrive/CICIDS_Dataset/cicids_cleaned.csv')
print("Loaded:", df.shape)

# ── Step 2: Split features and target ──
X = df.drop(columns=['target'])
y = df['target']

# ── Step 3: Train/test split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

# ── Step 4: Drop highly correlated features ──
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
X_train = X_train.drop(columns=drop_corr)
X_test = X_test.drop(columns=drop_corr)
print(f"After correlation drop: {X_train.shape}")

# ── Step 5: Drop low variance features ──
selector = VarianceThreshold(threshold=0.01)
selector.fit(X_train)
X_train = X_train[X_train.columns[selector.get_support()]]
X_test = X_test[X_test.columns[selector.get_support()]]
print(f"After variance threshold: {X_train.shape}")

# ── Step 6: Select top 40 features by RF importance ──
X_sample = X_train.sample(n=min(50000, len(X_train)), random_state=42)
y_sample = y_train[X_sample.index]

rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = importances.nlargest(40).index.tolist()
X_train = X_train[top_features]
X_test = X_test[top_features]
print(f"After feature selection: {X_train.shape}")

# ── Step 7: Scale ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ── Step 8: Build numpy arrays (features + target combined) ──
train_arr = np.c_[X_train_scaled, np.array(y_train)]
test_arr = np.c_[X_test_scaled, np.array(y_test)]
print("train_arr shape:", train_arr.shape)
print("test_arr shape:", test_arr.shape)

# ── Step 9: Save arrays and feature names to Drive ──
np.save('/content/drive/MyDrive/CICIDS_Dataset/train_arr.npy', train_arr)
np.save('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy', test_arr)

with open('/content/drive/MyDrive/CICIDS_Dataset/model/feature_names.pkl', 'wb') as f:
    pickle.dump(top_features, f)

with open('/content/drive/MyDrive/CICIDS_Dataset/model/preprocessor.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("All saved to Drive!")

Loaded: (836641, 101)
Train: (669312, 100) Test: (167329, 100)
After correlation drop: (669312, 62)
After variance threshold: (669312, 57)
After feature selection: (669312, 40)
train_arr shape: (669312, 41)
test_arr shape: (167329, 41)
All saved to Drive!


In [ ]:
import numpy as np
from networksecurity.utils.main_utils.utils import load_numpy_array_data
from networksecurity.components.anomaly_detector import AnomalyDetector

# Load your transformed arrays from Drive
train_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/train_arr.npy')
test_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy')

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]

# Train autoencoder
anomaly_detector = AnomalyDetector(
    input_dim=X_train.shape[1],
    epochs=20,
    batch_size=256,
    lr=0.001
)

results = anomaly_detector.initiate_anomaly_detector(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    save_path='final_model/autoencoder.pkl'
)

print(results)

{'threshold': 1.0181893110275269, 'accuracy': 0.6129003340723963, 'detection_rate': 0.0425037037037037, 'false_alarm_rate': 0.0014224323593344619}


In [ ]:
shutil.copy('final_model/autoencoder.pkl', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder.pkl')
print("Done!")

Done!


In [ ]:
import pickle
import torch

# Load the autoencoder
with open('final_model/autoencoder.pkl', 'rb') as f:
    autoencoder = pickle.load(f)

# Move everything to CPU
autoencoder.model = autoencoder.model.cpu()
autoencoder.device = torch.device('cpu')

# Re-save
with open('final_model/autoencoder_cpu.pkl', 'wb') as f:
    pickle.dump(autoencoder, f)

# Copy to Drive
import shutil
shutil.copy('final_model/autoencoder_cpu.pkl', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_cpu.pkl')
print("Done!")

Done!


In [ ]:
import pickle
import torch

# Load existing autoencoder
with open('final_model/autoencoder.pkl', 'rb') as f:
    autoencoder = pickle.load(f)

# Save PyTorch model weights separately using torch.save
torch.save(autoencoder.model.state_dict(), 'final_model/autoencoder_weights.pth')

# Save everything else (threshold, config) without the PyTorch model
autoencoder.model = None  # remove the PyTorch model from the object
with open('final_model/autoencoder_meta.pkl', 'wb') as f:
    pickle.dump(autoencoder, f)

# Copy both to Drive
import shutil
shutil.copy('final_model/autoencoder_weights.pth', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_weights.pth')
shutil.copy('final_model/autoencoder_meta.pkl', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_meta.pkl')
print("Done!")

Done!


In [ ]:
import pickle
import torch

# Load existing autoencoder
with open('final_model/autoencoder.pkl', 'rb') as f:
    autoencoder = pickle.load(f)

# Save weights to CPU
torch.save(
    autoencoder.model.cpu().state_dict(),
    'final_model/autoencoder_weights.pth'
)

# Save only non-PyTorch attributes
meta = {
    'threshold': autoencoder.threshold,
    'input_dim': autoencoder.input_dim,
    'epochs': autoencoder.epochs,
    'batch_size': autoencoder.batch_size,
    'lr': autoencoder.lr
}

with open('final_model/autoencoder_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

import shutil
shutil.copy('final_model/autoencoder_weights.pth', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_weights.pth')
shutil.copy('final_model/autoencoder_meta.pkl', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_meta.pkl')
print("Saved! Threshold:", autoencoder.threshold)

Saved! Threshold: 1.0181893110275269


In [16]:
import pickle
import numpy as np

# Load autoencoder
with open('final_model/autoencoder.pkl', 'rb') as f:
    autoencoder = pickle.load(f)

# Load test array
test_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy')
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]

# Get results
y_pred = autoencoder.predict(X_test)

attack_mask = y_test == 1
benign_mask = y_test == 0

detection_rate = np.mean(y_pred[attack_mask] == 1)
false_alarm_rate = np.mean(y_pred[benign_mask] == 1)

print(f"Threshold      : {autoencoder.threshold:.4f}")
print(f"Detection Rate : {detection_rate:.4f} ({detection_rate*100:.2f}%)")
print(f"False Alarm    : {false_alarm_rate:.4f} ({false_alarm_rate*100:.2f}%)")

Threshold      : 1.0182
Detection Rate : 0.0425 (4.25%)
False Alarm    : 0.0014 (0.14%)


In [4]:
import numpy as np
import pickle
from networksecurity.components.anomaly_detector import AnomalyDetector

# Load arrays
train_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/train_arr.npy')
test_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy')

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]

# Retrain with more epochs and lower threshold sensitivity
anomaly_detector = AnomalyDetector(
    input_dim=X_train.shape[1],
    epochs=50,          # increased from 20
    batch_size=256,
    lr=0.001,
)

results = anomaly_detector.initiate_anomaly_detector(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    save_path='final_model/autoencoder.pkl'
)

print(results)

{'threshold': 1.557409644126892, 'accuracy': 0.6073723024699843, 'detection_rate': 0.027274074074074076, 'false_alarm_rate': 0.00039066804235242267}


In [5]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle

train_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/train_arr.npy')
test_arr = np.load('/content/drive/MyDrive/CICIDS_Dataset/test_arr.npy')

X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
X_test, y_test = test_arr[:, :-1], test_arr[:, -1]

input_dim = X_train.shape[1]

# ── Improved Autoencoder ──
class ImprovedAutoEncoder(nn.Module):
    """
    Deeper architecture with BatchNorm and Dropout.
    BatchNorm stabilizes training.
    Dropout prevents overfitting.
    Deeper layers learn more complex patterns of normal traffic.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


# ── Training ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Train on benign only
X_benign = X_train[y_train == 0]
print(f"Training on {X_benign.shape[0]} benign samples")

X_tensor = torch.FloatTensor(X_benign).to(device)
dataset = TensorDataset(X_tensor, X_tensor)
loader = DataLoader(dataset, batch_size=512, shuffle=True)

model = ImprovedAutoEncoder(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

# Train for 100 epochs
for epoch in range(100):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in loader:
        reconstructed = model(batch_X)
        loss = criterion(reconstructed, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    scheduler.step(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/100] Loss: {avg_loss:.6f}")


# ── Find Best Threshold ──
model.eval()
with torch.no_grad():
    X_benign_tensor = torch.FloatTensor(X_benign).to(device)
    recon = model(X_benign_tensor)
    benign_errors = torch.mean((X_benign_tensor - recon) ** 2, dim=1).cpu().numpy()

print("\nFinding best threshold...")
for multiplier in [0.5, 1.0, 1.5, 2.0, 3.0]:
    threshold = np.mean(benign_errors) + multiplier * np.std(benign_errors)

    X_test_tensor = torch.FloatTensor(X_test).to(device)
    with torch.no_grad():
        recon_test = model(X_test_tensor)
        test_errors = torch.mean((X_test_tensor - recon_test) ** 2, dim=1).cpu().numpy()

    y_pred = (test_errors > threshold).astype(int)

    attack_mask = y_test == 1
    benign_mask = y_test == 0
    detection_rate = np.mean(y_pred[attack_mask] == 1)
    false_alarm = np.mean(y_pred[benign_mask] == 1)

    print(f"Multiplier: {multiplier:.1f} | Threshold: {threshold:.4f} | Detection: {detection_rate*100:.2f}% | False Alarm: {false_alarm*100:.2f}%")

Using device: cuda
Training on 399315 benign samples
Epoch [10/100] Loss: 0.151409
Epoch [20/100] Loss: 0.116131
Epoch [30/100] Loss: 0.106220
Epoch [40/100] Loss: 0.098248
Epoch [50/100] Loss: 0.099390
Epoch [60/100] Loss: 0.090565
Epoch [70/100] Loss: 0.085080
Epoch [80/100] Loss: 0.083910
Epoch [90/100] Loss: 0.082386
Epoch [100/100] Loss: 0.082477

Finding best threshold...
Multiplier: 0.5 | Threshold: 0.8841 | Detection: 23.15% | False Alarm: 1.49%
Multiplier: 1.0 | Threshold: 1.6658 | Detection: 3.02% | False Alarm: 0.52%
Multiplier: 1.5 | Threshold: 2.4474 | Detection: 0.09% | False Alarm: 0.36%
Multiplier: 2.0 | Threshold: 3.2291 | Detection: 0.02% | False Alarm: 0.29%
Multiplier: 3.0 | Threshold: 4.7924 | Detection: 0.02% | False Alarm: 0.15%


In [6]:
import pickle
import torch

# Set best threshold
best_threshold = 0.8841
model.eval()

# Save weights to CPU
torch.save(
    model.cpu().state_dict(),
    'final_model/autoencoder_weights.pth'
)

# Save meta
meta = {
    'threshold': best_threshold,
    'input_dim': input_dim,
    'epochs': 100,
    'batch_size': 512,
    'lr': 0.001
}

with open('final_model/autoencoder_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

import shutil
shutil.copy('final_model/autoencoder_weights.pth', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_weights.pth')
shutil.copy('final_model/autoencoder_meta.pkl', '/content/drive/MyDrive/CICIDS_Dataset/model/autoencoder_meta.pkl')
print("Saved with threshold:", best_threshold)

Saved with threshold: 0.8841
